In [ ]:
# Importations

import time
from enderscope import SerialUtils, Stage, Panel
import serial
from math import * 
import threading 

In [2]:
# Variables

rectangle = [40,30] # in mm, x and y

printer_speed = 3 # in mm/s
width_extrusion = 5 # in mm 
number_passages = min(ceil(rectangle[0] /(2 * width_extrusion)),ceil(rectangle[1] /(2 * width_extrusion)))  # computes the smallest integer that is greater than or equal to x.
number_layer = 1 # To modify 
layer_thickness = 2 # in mm

horizontal_offset = 12.86 # in mm, horizontal distance between two holes
vertical_offset = 13.6 # in mm, vertical distance between two holes

In [3]:
# Ports

ports = SerialUtils.serial_ports() # List of ports
print (ports) 

syringe_pump_port = ports[0] # To modify
printer_port = ports[1] # To modify

s = Stage(printer_port, 115200) 
syringe_pump = serial.Serial(port= syringe_pump_port, baudrate=115200, timeout=0.01, writeTimeout=1) # Connexion syringe pump

['COM4', 'COM5']


In [4]:
# Syringe pump : messages

def message_start(letter_syringe_pump): 
    return(f"{letter_syringe_pump}\n".encode('utf8')) 

def message_stop():
        return b"S\n"

In [5]:
# Purge 

def purge_syringe_pump(letter_syringe_pump):
    syringe_pump.write(message_start(letter_syringe_pump))
    time.sleep(8)  # time for the purge 
    syringe_pump.write(message_stop())
    time.sleep(1)

purge_A = threading.Thread( target = purge_syringe_pump, args = ('A'))

def purge():
    purge_A.start()
    purge_A.join()

In [6]:
# Rectangle

def function_rectangle(): 

    x_rectangle = rectangle[0]
    y_rectangle = rectangle[1]

    for i in range (number_passages):
        s.write_code(f"M203 X{printer_speed}")
        s.write_code(f"M203 Y{printer_speed}")
        s.move_axis('x', x_rectangle)
        s.move_axis ('y', y_rectangle)
        s.move_axis('x', -x_rectangle)
        s.move_axis('y', - (y_rectangle - width_extrusion))

        

        s.move_axis('x', width_extrusion)
        
        x_rectangle = x_rectangle - (2 * width_extrusion) # The next inner rectangle will have two fewer layers
        y_rectangle = y_rectangle - (2*width_extrusion)

    s.write_code(f"M400")

In [7]:
# Go to the position and adjust according to the syringe pump 

def go (letter_syringe_pump):
    s.write_code(f"M203 X{printer_speed}")
    s.write_code(f"M203 Y{printer_speed}")
    if letter_syringe_pump == 'A':
        pass

    if letter_syringe_pump =='B':
        s.move_axis('x', rectangle[0] - horizontal_offset)   # a horizontal distance equal to 'horizontal_offset' between A and B
        s.move_axis('z', - (number_layer - 1) * layer_thickness) # Return to the initial position in z 
        s.write_code(f"M400")

    if letter_syringe_pump == 'C':
        s.move_axis('x', rectangle[0] + horizontal_offset) 
        s.move_axis('y', - vertical_offset) # a vertical distance equal to 'vertical_offset' betwween B and C 
        s.move_axis('z', - (number_layer - 1) * layer_thickness) # Return to the initial position in z 
        s.write_code(f"M400")

In [8]:
# Go to the initial position

def go_initial_position():
    s.write_code(f"M203 X{printer_speed}")
    s.write_code(f"M203 Y{printer_speed}")
    s.move_axis('x', - width_extrusion * number_passages)  # Return to the initial position in x
    s.move_axis('y',- width_extrusion * number_passages)   # Return to the initial position in y
    s.write_code(f"M400")
    

In [9]:
# Print rectangle 

def print_rectangle(letter_syringe_pump):
    go(letter_syringe_pump)

    for i in range(number_layer): # Repeat for each layer 
        if i > 1:
            s.move_axis('z', layer_thickness) # With each new layer, the height is increased 
        syringe_pump.write(message_start(letter_syringe_pump)) 
        function_rectangle() 
        syringe_pump.write(message_stop())
        go_initial_position() 

In [11]:
# Choose the purge point FOR A (at the bottom left)
#      ________
#   C | °    ° | D
#   A | °    ° | B  
#      ¯¯¯¯¯¯¯¯
p = Panel(s) 

GridspecLayout(children=(Button(description='Up', layout=Layout(grid_area='widget001', height='auto', width='a…

Output()

In [12]:
# Purge 
purge()

In [13]:
# Choose the printing location
p = Panel(s) 

GridspecLayout(children=(Button(description='Up', layout=Layout(grid_area='widget001', height='auto', width='a…

Output()

In [14]:
# Print the rectangles
print_rectangle('A')
print_rectangle('B')
print_rectangle('C')